# ⚖️ THEMIS — Indian Legal Intelligence Engine

**The Parametric Legal Intelligence Engine for Indian Law**

![Model](https://img.shields.io/badge/model-Mistral%207B%20LoRA-blueviolet)
![Domain](https://img.shields.io/badge/domain-Indian%20Law%20%7C%20BNS%20%7C%20IPC-crimson)
![Python](https://img.shields.io/badge/python-3.11+-brightgreen)
![License](https://img.shields.io/badge/license-MIT-lightgrey)

---

## What is this notebook?

This notebook runs **THEMIS** — a domain-specific LLM fine-tuned on Indian statutory law using **LoRA (Low-Rank Adaptation)**. It answers legal questions about:

- **Bharatiya Nyaya Sanhita (BNS) 2023** — Criminal law
- **Bharatiya Nagarik Suraksha Sanhita (BNSS) 2023** — Criminal procedure
- **Indian Penal Code (IPC) 1860** — Legacy criminal law
- **Consumer Protection Act 2019** — Consumer rights
- **Right to Information Act 2005** — RTI

| Component | Detail |
|-----------|--------|
| Base Model | `unsloth/mistral-7b-instruct-v0.3-bnb-4bit` |
| LoRA Adapter | `Daniel2503/themis-mistral-7b-lora` |
| Training Data | 1,939 Indian legal Q&A pairs |
| LoRA Config | rank=8, alpha=16, targets: q_proj, v_proj |
| Quantization | 4-bit NF4 (bitsandbytes) |
| Fine-tuned with | Unsloth on Kaggle T4 GPU |

**Important:** This provides legal *orientation*, not legal * advice. Always consult a qualified advocate.

---

## Setup Instructions (Kaggle)

1. **Enable GPU:** Go to ⚙️ Settings → Accelerator → **GPU T4 × 2** (or T4 × 1)
2. **Enable Internet:** Go to ⚙️ Settings → Internet → **Turn on** (needed to download the model from HuggingFace)
3. **Run All:** Runtime → Run all cells
4. The model (~4GB) will download automatically on the first run and cache for subsequent runs

---

## Cell 1 — Install Dependencies

Installs the required packages: `transformers`, `peft`, `bitsandbytes`, `accelerate`, and `rich`.

In [ ]:
!pip install -q transformers>=4.35.0 peft>=0.6.0 bitsandbytes>=0.41.0 accelerate>=0.21.0 rich>=13.0.0

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Settings \u2192 Accelerator \u2192 GPU T4")

---
## Cell 2 — Configuration

All model IDs, paths, and generation parameters in one place.

In [ ]:
# === Model IDs ===
BASE_MODEL = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"  # 4-bit quantized Mistral 7B
LORA_ADAPTER = "Daniel2503/themis-mistral-7b-lora"          # Fine-tuned LoRA weights

# === Generation Parameters ===
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.3        # Low = more deterministic, High = more creative
TOP_P = 0.9
REPETITION_PENALTY = 1.1

# === Prompt Format (Alpaca-style) ===
INSTRUCTION_TEMPLATE = """### Instruction:
{question}

### Response:
"""

# === Disclaimer ===
DISCLAIMER = (
    "\n\nDISCLAIMER: This is legal orientation, not legal advice. "
    "Consult a qualified advocate for your specific situation."
)

print("Configuration loaded.")
print(f"Base model:   {BASE_MODEL}")
print(f"LoRA adapter: {LORA_ADAPTER}")
print(f"Temperature:  {TEMPERATURE}")
print(f"Max tokens:   {MAX_NEW_TOKENS}")

---
## Cell 3 — Load Model + LoRA Adapter

This is the core cell. It:
1. Sets up **4-bit quantization** (NF4) via BitsAndBytes — compresses the 14GB model to ~4GB
2. Downloads and loads the **base model** from HuggingFace Hub
3. Downloads and attaches the **LoRA adapter** on top of the base model
4. Merges the adapter into the base model for faster inference

**First run:** Downloads ~4GB (base) + ~33MB (adapter). Cached for subsequent runs.
**Time:** ~2-3 minutes on T4 GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Step 1: 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Step 2: Load tokenizer
print(f"Loading tokenizer from {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")

# Step 3: Load base model in 4-bit
print(f"\nLoading base model: {BASE_MODEL}")
print("(This downloads ~4GB on first run, cached afterward)\n")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
print(f"Base model loaded on {base_model.device}")

# Step 4: Attach LoRA adapter
print(f"\nLoading LoRA adapter: {LORA_ADAPTER}")
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER)
model.eval()
print("LoRA adapter attached.")

# Step 5 (Optional): Merge adapter into base for faster inference
# This fuses LoRA weights permanently — slightly faster but uses more VRAM
# model = model.merge_and_unload()

print("\n✅ Model ready for inference!")
print(f"   Device: {model.device}")
print(f"   VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

---
## Cell 4 — Inference Function

Defines the `ask()` function that:
1. Takes a plain English legal question
2. Formats it as `### Instruction:\n{question}\n\n### Response:\n`
3. Generates a response with the loaded model
4. Returns the response text (stripped of the prompt)

In [ ]:
def ask(question: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    """
    Ask a legal question and get a response.
    
    Args:
        question: Plain English legal question
        max_new_tokens: Maximum tokens to generate
    
    Returns:
        Response text with disclaimer
    """
    # Format prompt
    prompt = INSTRUCTION_TEMPLATE.format(question=question)
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_tokens = inputs["input_ids"].shape[1]
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only new tokens (strip the prompt)
    new_tokens = outputs[0][input_tokens:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    # Add disclaimer if not present
    if "disclaimer" not in response.lower():
        response += DISCLAIMER
    
    return response

print("ask() function defined.")

---
## Cell 5 — Single-Shot Q&A

Run the cell below, type your legal question, and press Enter.

### Example questions to try:
- "What is the punishment for theft under BNS?"
- "How do I file an RTI application?"
- "What are my rights as a consumer if a product is defective?"
- "What is Section 302 of IPC and its BNS equivalent?"
- "Can my employer refuse to pay my salary?"

In [ ]:
# @title ⚖️ Ask a Legal Question
question = "What is the punishment for theft under BNS?"  # @param {type:"string"}

print(f"❓ Question: {question}\n")
print("Thinking...\n")

response = ask(question)

print("=" * 60)
print("LEGAL ORIENTATION")
print("=" * 60)
print(response)
print("=" * 60)

---
## Cell 6 — Batch Evaluation

Run multiple questions at once and collect responses. Useful for testing and demo purposes.

In [ ]:
# @title Run Batch Evaluation
test_questions = [
    "What is the punishment for murder under BNS 2023?",
    "What is the difference between BNS and IPC?",
    "How does the Consumer Protection Act help buyers?",
    "What are the provisions for bail under BNSS?",
    "How do I file an RTI application in India?",
]

print(f"Running {len(test_questions)} evaluation questions...\n")

results = []
for i, q in enumerate(test_questions, 1):
    print(f"\n--- Q{i}: {q}")
    print("-" * 50)
    try:
        response = ask(q)
        print(response)
        results.append({"question": q, "response": response, "status": "ok"})
    except Exception as e:
        print(f"ERROR: {e}")
        results.append({"question": q, "response": str(e), "status": "error"})

print(f"\n\n✅ Completed {len(results)} questions.")
print(f"   Successful: {sum(1 for r in results if r['status'] == 'ok')}")
print(f"   Failed: {sum(1 for r in results if r['status'] == 'error')}")

---
## Cell 7 — Interactive Chat Mode

Multi-turn conversation with memory. Ask follow-up questions.

Type `exit`, `quit`, or `q` to end the session.

In [ ]:
# @title 🗣️ Interactive Chat
print("=" * 60)
print("  THEMIS Interactive Legal Q&A")
print("  Type 'exit' to quit")
print("=" * 60)
print()

chat_history = []

while True:
    try:
        question = input("❓ Ask: ")
    except (EOFError, KeyboardInterrupt):
        print("\nGoodbye!")
        break
    
    if question.strip().lower() in ("exit", "quit", "q", ""):
        print("Goodbye!")
        break
    
    print("\nThinking...\n")
    
    try:
        response = ask(question)
        print("=" * 60)
        print(response)
        print("=" * 60)
        
        # Store in history
        chat_history.append({"role": "user", "content": question})
        chat_history.append({"role": "assistant", "content": response})
    except Exception as e:
        print(f"Error: {e}")
    
    print()

print(f"\nSession ended. {len(chat_history) // 2} exchanges recorded.")

---
## Cell 8 — Save Responses to File (Optional)

Export the batch evaluation results as a JSON file for your portfolio or records.

In [ ]:
import json
from datetime import datetime

# @title Export Results
output = {
    "model": LORA_ADAPTER,
    "base_model": BASE_MODEL,
    "timestamp": datetime.now().isoformat(),
    "config": {
        "temperature": TEMPERATURE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "top_p": TOP_P,
    },
    "results": results if 'results' in dir() else [],
    "chat_history": chat_history if 'chat_history' in dir() else [],
}

filename = f"themis_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(filename, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Results saved to {filename}")

---

## Performance

| Metric | T4 GPU (this notebook) | CPU (no GPU) |
|--------|----------------------|--------------|
| Model download | ~2 min | ~10 min |
| Model load | ~30 sec | ~5 min |
| First token latency | ~1 sec | ~30 sec |
| Full response (512 tokens) | ~5-8 sec | ~3-5 min |
| VRAM/RAM usage | ~4.5 GB VRAM | ~14 GB RAM |

---

## Project Links

| Resource | Link |
|----------|------|
| LoRA Adapter | [Daniel2503/themis-mistral-7b-lora](https://huggingface.co/Daniel2503/themis-mistral-7b-lora) |
| GitHub | [github.com/Daniel2503/themis](https://github.com/Daniel2503/themis) |
| License | MIT |

---

## How It Works

```
User Question
    |
    v
Prompt Formatting (### Instruction / ### Response)
    |
    v
Mistral 7B (4-bit quantized) + LoRA Adapter
    |
    v
Generated Response (with section citations)
    |
    v
Disclaimer appended
```

**THEMIS is NOT:** a retrieval system, a chatbot wrapper, or a lawyer substitute. It is a parametric knowledge model — legal understanding is baked into the weights through supervised fine-tuning on 1,939 Indian legal Q&A pairs.

---

## License

MIT License — See [LICENSE](https://github.com/Daniel2503/themis/blob/main/LICENSE) for details.

---

*THEMIS — Greek goddess of law, justice, and order. Because justice should not require a law degree to understand.*